# Household Power Consumption - Data Exploration

This notebook explores the household power consumption dataset, performing data loading, cleaning, visualization, and statistical analysis.

## 1. Import Required Libraries

In [ ]:
import sys
from pathlib import Path

# Add parent directory to path
sys.path.insert(0, str(Path.cwd().parent))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime

# Set up plotting
%matplotlib inline
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (14, 6)

print('✓ All libraries imported successfully')

## 2. Load the Raw Data

In [ ]:
# Load data from src module
from src.data.loader import load_raw_data
from src.utils.config import DATA_RAW_PATH

# Load the household power consumption data
data_path = DATA_RAW_PATH / 'household_power_consumption.txt'
print(f'Loading data from: {data_path}')

try:
    df = load_raw_data(str(data_path))
    print(f'✓ Data loaded successfully!')
    print(f'\nDataset shape: {df.shape}')
    print(f'Memory usage: {df.memory_usage(deep=True).sum() / 1024**2:.2f} MB')
except Exception as e:
    print(f'✗ Error loading data: {e}')
    print('\nMake sure you have placed household_power_consumption.txt in data/raw/ directory')

## 3. Display First Few Rows

In [ ]:
df.head(10)

## 4. Data Information and Statistics

In [ ]:
print('Data Types:')
print(df.dtypes)
print('\n' + '='*60)
print('\nBasic Statistics:')
df.describe()

## 5. Check for Missing Values

In [ ]:
# Missing values
missing = df.isnull().sum()
missing_pct = (df.isnull().sum() / len(df)) * 100

missing_df = pd.DataFrame({
    'Missing_Count': missing,
    'Percentage': missing_pct
})

missing_df = missing_df[missing_df['Missing_Count'] > 0].sort_values('Percentage', ascending=False)

print('Missing Values Analysis:')
print(missing_df)
print(f'\nTotal missing values: {df.isnull().sum().sum()}')

## 6. Visualize Global Active Power Over Time

In [ ]:
plt.figure(figsize=(16, 6))
plt.plot(df['datetime'], df['Global_active_power'], linewidth=0.5, alpha=0.8)
plt.title('Global Active Power Over Time', fontsize=14, fontweight='bold')
plt.xlabel('Date', fontsize=12)
plt.ylabel('Power (kW)', fontsize=12)
plt.xticks(rotation=45)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print(f'Global Active Power - Min: {df["Global_active_power"].min():.2f} kW')
print(f'Global Active Power - Max: {df["Global_active_power"].max():.2f} kW')
print(f'Global Active Power - Mean: {df["Global_active_power"].mean():.2f} kW')

## 7. Distribution Analysis

In [ ]:
# Select key columns for distribution analysis
columns_to_analyze = ['Global_active_power', 'Global_reactive_power', 'Voltage', 'Global_intensity']

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle('Distribution of Key Features', fontsize=16, fontweight='bold')

for idx, col in enumerate(columns_to_analyze):
    ax = axes[idx // 2, idx % 2]
    ax.hist(df[col].dropna(), bins=50, edgecolor='black', alpha=0.7, color='steelblue')
    ax.set_title(f'{col} Distribution')
    ax.set_xlabel(col)
    ax.set_ylabel('Frequency')
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 8. Correlation Analysis

In [ ]:
# Select numeric columns
numeric_cols = df.select_dtypes(include=[np.number]).columns
correlation_matrix = df[numeric_cols].corr()

plt.figure(figsize=(10, 8))
sns.heatmap(correlation_matrix, annot=True, cmap='coolwarm', center=0, fmt='.2f', 
            square=True, cbar_kws={'label': 'Correlation'})
plt.title('Feature Correlation Matrix', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

# Show correlations with Global_active_power
print('\nCorrelation with Global_active_power:')
correlations = df[numeric_cols].corr()['Global_active_power'].sort_values(ascending=False)
print(correlations)

## 9. Sub-metering Analysis

In [ ]:
# Analyze sub-metering data
submeter_cols = ['Sub_metering_1', 'Sub_metering_2', 'Sub_metering_3']

fig, axes = plt.subplots(1, 3, figsize=(16, 5))
fig.suptitle('Sub-metering Analysis', fontsize=16, fontweight='bold')

for idx, col in enumerate(submeter_cols):
    ax = axes[idx]
    ax.plot(df['datetime'][:1000], df[col][:1000], linewidth=0.8, alpha=0.8)
    ax.set_title(f'{col} (First 1000 points)')
    ax.set_xlabel('Date')
    ax.set_ylabel('Energy (Wh)')
    ax.grid(True, alpha=0.3)
    plt.setp(ax.xaxis.get_majorticklabels(), rotation=45)

plt.tight_layout()
plt.show()

# Statistics
print('Sub-metering Statistics:')
for col in submeter_cols:
    print(f'\n{col}:')
    print(f'  Mean: {df[col].mean():.2f} Wh')
    print(f'  Max: {df[col].max():.2f} Wh')
    print(f'  Min: {df[col].min():.2f} Wh')

## 10. Data Quality Summary

In [ ]:
print('='*60)
print('DATA QUALITY SUMMARY')
print('='*60)
print(f'\nDataset Shape: {df.shape}')
print(f'Date Range: {df["datetime"].min()} to {df["datetime"].max()}')
print(f'Total Records: {len(df):,}')
print(f'Total Features: {len(df.columns)}')
print(f'\nMissing Values: {df.isnull().sum().sum()} ({(df.isnull().sum().sum() / (df.shape[0] * df.shape[1]) * 100):.2f}%)')
print(f'\nMemory Usage: {df.memory_usage(deep=True).sum() / 1024**2:.2f} MB')
print(f'\nDuplicate Rows: {df.duplicated().sum()}')
print('\n' + '='*60)

## 11. Next Steps

With this exploration, we have:

✓ Loaded and inspected the raw data  
✓ Identified missing values and data quality issues  
✓ Visualized power consumption patterns  
✓ Analyzed correlations between features  
✓ Examined sub-metering data  

**Recommended Next Steps:**
1. Run `python train_pipeline.py` to preprocess and engineer features
2. Train ML models to predict power consumption
3. Evaluate model performance and visualize predictions
4. Deploy the model using Flask API